# Apriori — recurring co-occurrence patterns

Discover frequent observed associations. With at most 60 cases, rules are exploratory and unstable.

**Real data only.** Run ingestion first. Never interpret an empty or skipped model as a finding.

In [1]:
from pathlib import Path
import sys
for loc in [Path.cwd(),Path.cwd()/'notebooks',Path.cwd()/'research_expansion'/'notebooks']:
    if (loc/'common.py').exists():
        sys.path.insert(0,str(loc.resolve()));break
from common import root, load_panel, load_cases, output

In [2]:
import itertools,collections,pandas as pd
c=load_cases().dropna(subset=['delta_pp','shock_std_yoy','negative_share']).copy()
if len(c)<12:raise ValueError('At least 12 observed cases needed for association rules')
vol_median=c.shock_std_yoy.median(); neg_median=c.negative_share.median()
transactions=[]
for _,r in c.iterrows():
    transactions.append(frozenset([f"country={r['country']}",f"event={r['event']}",
        f"industry={r['industry_code']}",'slowdown=yes' if r['delta_pp']<0 else 'slowdown=no',
        'volatility=high' if r['shock_std_yoy']>vol_median else 'volatility=low',
        'negative_share=high' if r['negative_share']>neg_median else 'negative_share=low']))
N=len(transactions);support=lambda item:sum(item.issubset(t) for t in transactions)/N
min_support=max(0.12,4/N)
# True Apriori join/prune: only grow candidates whose every (k-1) subset was frequent.
freq={frozenset([i]):support(frozenset([i])) for i in sorted(set().union(*transactions))}
freq={i:v for i,v in freq.items() if v>=min_support}
all_frequent=dict(freq)
for k in (2,3):
    prev=set(freq)
    candidates=set()
    for a,b in itertools.combinations(prev,2):
        x=a|b
        if len(x)==k and all(frozenset(s) in prev for s in itertools.combinations(x,k-1)):
            candidates.add(x)
    freq={it:support(it) for it in candidates if support(it)>=min_support}
    all_frequent.update(freq)
rules=[]
for items,sab in all_frequent.items():
    if len(items)<2:continue
    for k in range(1,len(items)):
        for left_ in itertools.combinations(items,k):
            A=frozenset(left_);B=items-A
            sa=support(A);sb=support(B)
            conf=sab/sa if sa else 0; lift=conf/sb if sb else 0
            if conf>=0.6 and lift>1:
                rules.append(dict(antecedent=sorted(A),consequent=sorted(B),
                                 support=sab,confidence=conf,lift=lift))
rules=sorted(rules,key=lambda r:(-r['lift'],-r['support']))[:30]
display(pd.DataFrame(rules).head(12))
output('apriori',{'model':'Apriori','status':'fitted','n_cases':N,'min_support':min_support,
'n_frequent_itemsets':len(all_frequent),'rules':rules,
'limitation':'Descriptive co-occurrence with a small number of non-independent country-industry-event cases; not cause, prediction or statistical significance.'})

,antecedent,consequent,support,confidence,lift
0,[event=supply_chain_2021],"[slowdown=no, volatility=high]",0.130,0.866667,3.611111
1,"[event=supply_chain_2021, volatility=high]",[slowdown=no],0.130,0.962963,2.048857
2,[event=supply_chain_2021],[slowdown=no],0.140,0.933333,1.985816
3,"[event=supply_chain_2021, slowdown=no]",[volatility=high],0.130,0.928571,1.857143
4,[event=supply_chain_2021],[volatility=high],0.135,0.900000,1.800000
5,"[country=DEU, slowdown=yes]",[negative_share=high],0.145,0.805556,1.732378
6,[event=covid_disruption_2020],[volatility=high],0.125,0.833333,1.666667
7,"[country=USA, slowdown=no]",[negative_share=low],0.160,0.888889,1.661475
8,"[country=USA, negative_share=low]",[slowdown=no],0.160,0.780488,1.660612
9,"[country=DEU, negative_share=low]",[slowdown=no],0.120,0.774194,1.647220


Saved C:\Users\mehak\OneDrive\ドキュメント\ChatGPT\Industry Res\reports\research\models\apriori.json


WindowsPath('C:/Users/mehak/OneDrive/ドキュメント/ChatGPT/Industry Res/reports/research/models/apriori.json')